# Clase 2 — De chatbot a agente con un LLM local

En la Clase 1 distinguimos un chatbot de un agente. Ahora vamos a construir el componente que interpreta lenguaje: un LLM local.

Ya ejecutamos un LLM local con llama.cpp en el Módulo 1 y aprendimos a escribir prompts en el Módulo 2. Esta vez el objetivo no es repetir la instalación, sino cambiar la pregunta:

> ¿Cómo envolvemos un modelo conversacional para que sea una pieza controlable de un agente?

## Objetivos

- Conectar el trabajo anterior: tokens, contexto, cuantización e instrucciones.
- Construir una única función para consultar el LLM local.
- Diferenciar una respuesta libre de una respuesta con objetivo y límites.
- Definir el contrato que compartirán el agente por reglas y el agente con LLM.
- Consolidar todo en un único ejercicio realista: un agente de triage para mesa de ayuda.
- Reutilizar el modelo descargado en la Clase 1 (o descargarlo la primera vez) y ejecutar todo con el LLM local real.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Un modelo no es un agente |
| 2 | El modelo de hoy: LFM2.5 1.2B |
| 3 | El caso realista: triage de tickets |
| 4 | Cargar el modelo una sola vez |
| 5 | El wrapper estable y su contrato |
| 6 | Respuesta libre contra respuesta controlada |
| 7 | El contrato común de los dos agentes |
| 8 | El agente de triage completo |
| 9 | Ejercicio: el agente de tu equipo |
| 10 | Resumen |

---
## 1. Un modelo no es un agente

Un LLM recibe mensajes y genera texto. No conoce por sí mismo el objetivo de nuestra mesa de ayuda, qué herramientas existen ni qué acciones están prohibidas.

El agente agregará esas decisiones alrededor del modelo:

    consulta
       ↓
    instrucciones + contexto
       ↓
    LLM local
       ↓
    salida validada
       ↓
    decisión permitida o revisión humana

Hoy construiremos el bloque LLM y su interfaz, y lo integraremos en un agente de triage que decide: **responder**, **pedir más datos** o **derivar a una persona**.

> **Puente con las clases anteriores:** en el Módulo 1 vimos que un LLM convierte texto en tokens y que la cuantización reduce memoria; en el Módulo 2 vimos que un prompt se compone de rol, objetivo, límites y formato. Hoy usaremos ambas ideas: el modelo se carga una sola vez (tokens y contexto) y la instrucción incluye límites y formato (anatomía del prompt).

---
## 2. El modelo de hoy: LFM2.5 1.2B Instruct

Usaremos **LFM2.5 1.2B Instruct** en formato GGUF cuantizado, publicado por Unsloth a partir del modelo de Liquid AI.

Datos técnicos relevantes:

| Concepto | Valor en esta práctica |
|---|---|
| Modelo instruct | Fine-tune de Liquid AI para seguir instrucciones y conversar |
| Parámetros | 1.17B (≈ 1170 millones) |
| Contexto | 32.768 tokens (ventana amplia, pero el contexto útil sigue siendo limitado) |
| GGUF | Formato listo para llama.cpp |
| Q8_0 | Cuantización de 8 bits del fichero elegido (≈ 1,25 GB) |
| llama.cpp | Motor que ejecuta el modelo en CPU/GPU local |
| ChatML | Formato conversacional que usa `<|im_start|>` / `<|im_end|>` |
| Soporte de idiomas | Incluye español |

La línea `unsloth/...` es el repositorio en Hugging Face que contiene los ficheros GGUF: conviene descargar el de menor tamaño y suficiente calidad. En clase usaremos `Q8_0` para maximizar calidad sobre memoria.

> **¿Hay que descargarlo de nuevo?** Este es el mismo `REPO_ID` y `FILENAME` de la **Clase 1 de este módulo**. Si ya la corriste, el archivo quedó en la caché de Hugging Face (≈ 1,25 GB) y esta clase lo reutiliza sin descargar. Si estás en una máquina nueva, la primera ejecución hará la descarga. El modelo del Módulo 1 (Qwen 2.5 0.5B) es otro modelo: no lo reutilizamos acá.

> **Recordatorio del Módulo 1:** la cuantización reduce memoria y permite trabajar en computadoras modestas, aunque puede disminuir algo la calidad. `Q8_0` conserva casi toda la calidad del modelo original.

In [ ]:
from pathlib import Path
import time

REPO_ID = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME = "LFM2.5-1.2B-Instruct-Q8_0.gguf"

print("Modelo:", REPO_ID)
print("Archivo:", FILENAME)

---
## 3. El caso realista: clasificador de tickets

Trabajaremos con un modelo útil de principio a fin: **HelpDesk**, una mesa de ayuda interna de una empresa de software.

El objetivo del agente es **clasificar** consultas de soporte: leer el mensaje y decidir una de tres acciones.

| Acción | Cuándo |
|---|---|
| responder | hay evidencia suficiente y la consulta es rutinaria |
| pedir_dato | falta un dato necesario (versión, número de ticket, mensaje de error) |
| derivar | riesgo financiero, datos sensibles, acción destructiva o consulta fuera del alcance |

Restricciones de la mesa de ayuda (contrato operativo):

- no ejecutar ninguna acción;
- no solicitar datos sensibles (contraseñas, tarjetas);
- no inventar procedimientos;
- la derivación a humano está permitida y es recomendada cuando corresponde.

Las consultas vienen del dataset realista `tickets_soporte.csv` que usaremos para comparar en la Clase 3; hoy tomamos una muestra representativa.

In [ ]:
import time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

inicio = time.time() # captura el tiempo de inicio para medir la duración de la carga del modelo

# `hf_hub_download` usa la caché local: si el archivo ya quedó de la
# Clase 1 de este módulo, reutiliza la copia y no vuelve a descargar.
ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

# Contexto 4096: suficiente para triage de tickets y muy por debajo
# de la ventana máxima de 32.768 tokens del modelo.
llm = Llama(
            model_path=ruta_modelo, 
            n_ctx=4096,
            n_gpu_layers=0, 
            verbose=False
            )
print(f"Modelo cargado en {time.time() - inicio:.1f} segundos")
print("Archivo local:", ruta_modelo)

---
## 4. Cargar el modelo una sola vez

Cargar el modelo dentro de cada consulta sería muy lento. Por eso se carga una vez y se conserva en la variable `llm`.

En la **Clase 1 de este mismo módulo** ya descargamos este modelo con `hf_hub_download`. Los archivos quedan en la caché de Hugging Face, así que:

- si ya corriste la Clase 1 con este mismo `REPO_ID` y `FILENAME`, **no se descarga de nuevo**: se reutiliza el archivo local;
- si es la primera vez en tu máquina, la celda descarga el archivo (~1,25 GB) y lo deja en caché para las próximas clases;
- el modelo del Módulo 1 (Qwen 2.5 0.5B) es **otro modelo**: no sirve para esta clase, acá usamos LFM2.5 1.2B.

La celda **siempre descarga si falta** y carga el archivo del modelo en `llm`; si ya lo tenés en caché (Clase 1), lo reutiliza al instante. Sin archivo local o sin conexión, la clase no puede inferir: el LLM local es obligatorio.

## 5. El wrapper: una interfaz estable

El resto del agente no debería conocer detalles de llama.cpp. Solo necesita una función con entradas claras.

Nuestro wrapper recibe:

- una consulta;
- una instrucción del sistema (system prompt);
- temperatura y longitud máxima de respuesta.

Devuelve la respuesta generada y metadatos (`modo`, `duracion_s`). El resto del agente nunca toca detalles de llama.cpp.

In [ ]:
def consultar_llm(consulta, 
                  instruccion, 
                  temperature=0.2,
                  max_tokens=160):
    if not isinstance(consulta, str) or not consulta.strip():
        return {"ok": False, "error": "consulta vacía", "modo": "validacion"}

    if llm is None:
        return {"ok": False, "error": "LLM no disponible", "modo": "sin_modelo"}

    inicio = time.time()
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": instruccion},
            {"role": "user", "content": consulta},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return {
        "ok": True,
        "pregunta": consulta,
        "respuesta": salida["choices"][0]["message"]["content"].strip(),
        "modo": REPO_ID,
        "duracion_s": round(time.time() - inicio, 2),
    }

In [ ]:
resultado = consultar_llm(
    consulta="La aplicación se cierra cuando exporto un reporte.",
    instruccion="Respondé libremente.",
)
resultado

El campo `modo` acompaña siempre la respuesta:

- `llm_local` significa que el modelo generó la respuesta;
- `validacion` o `sin_modelo` significa que el proceso se detuvo antes de inferir.

Esta trazabilidad será necesaria cuando comparemos agentes y documentemos riesgos (Clase 9 e ISO 42001).

---
## 6. Respuesta libre contra respuesta controlada

Una respuesta libre puede sonar útil, pero no necesariamente respeta el proceso de soporte. Aplicamos la anatomía del prompt del Módulo 2:

1. **rol**: componente de lenguaje de mesa de ayuda;
2. **objetivo**: orientar sin inventar procedimientos;
3. **límites**: no pedir contraseñas, no ejecutar acciones, no inventar;
4. **formato esperado**: secciones observables y accionables.

No estamos convirtiendo mágicamente al modelo en un agente. Estamos reduciendo la libertad de un componente probabilístico.

In [ ]:
CONSULTA = "La aplicación se cierra cuando exporto un reporte."

INSTRUCCION_LIBRE = "Respondé la consulta."

INSTRUCCION_CONTROLADA = """
Sos el componente de lenguaje de una mesa de ayuda.
Tu objetivo es orientar sin inventar procedimientos.
No solicites contraseñas ni ejecutes acciones.
Si falta información, pedí un solo dato adicional.
Respondé con: RESUMEN, SIGUIENTE PASO y REQUIERE HUMANO.
""".strip()

libre = consultar_llm(CONSULTA, INSTRUCCION_LIBRE)
controlada = consultar_llm(CONSULTA, INSTRUCCION_CONTROLADA)

display("RESPUESTA LIBRE", libre)
display("RESPUESTA CONTROLADA", controlada)

### Qué cambió

La respuesta controlada:

- evita una acción destructiva;
- reconoce que todavía faltan datos;
- produce secciones observables;
- permite que otro bloque del programa tome una decisión.

Un prompt ayuda, pero no garantiza cumplimiento. En la Clase 3 validaremos salidas estructuradas con Python.

---
## 7. El contrato común de los dos agentes

Para comparar reglas y LLM (Clase 3) necesitamos que ambos devuelvan los mismos campos. El contrato no indica cómo pensó el agente; indica qué información debe entregar al resto del sistema.

En esta clase el agente genera **solo interpretación** (triage); la herramienta y la acción llegarán en la Clase 4.

In [ ]:
def crear_resultado(categoria="sin_clasificar", decision="responder",
                    herramienta=None, respuesta="", confianza=None,
                    requiere_revision=False, traza=None):

    # Armamos un diccionario con la información de la respuesta del agente.
    return {
        "categoria": categoria,
        "decision": decision,          # responder | pedir_dato | derivar
        "herramienta": herramienta,    # se agregará en la Clase 4
        "respuesta": respuesta,
        "confianza": confianza,
        "requiere_revision": requiere_revision,
        "traza": traza or [],
    }

primer_resultado = crear_resultado(
    respuesta=controlada["respuesta"],
    traza=[{"paso": "generar_respuesta", "modo": controlada["modo"]}],
)
primer_resultado

---
## 8. Ejemplo con un agente de derivacion

Ahora combinamos todo: el agente interpreta la consulta con una instrucción controlada para HelpDesk y devuelve el contrato. Si el modelo falta o falla, no inventa una respuesta alternativa: lo declara y deriva.

La salida `RESUMEN / SIGUIENTE PASO / REQUIERE HUMANO` se mapea a la decisión del contrato. Por ahora el mapeo es textual (`.lower()` + condicionales); en la Clase 3 validaremos salidas JSON.

In [ ]:
def agente_triage(consulta):
    salida = consultar_llm(consulta, INSTRUCCION_CONTROLADA)
    if not salida["ok"]:
        return crear_resultado(
            decision="derivar",
            respuesta="El modelo no está disponible. Derivo a una persona.",
            requiere_revision=True,
            traza=[{"paso": "llm", "estado": "error",
                    "detalle": salida["error"]}],
        )

    # Normalizamos el texto: minúsculas y espacios a guion bajo
    # para que "REQUIERE HUMANO" coincida con "requiere_humano".
    texto = salida["respuesta"].lower().replace(" ", "_")
    if "requiere_humano" not in texto:
        decision, requiere_revision = "derivar", True
    else:
        resto = texto.split("requiere_humano", 1)[1][:20]
        if "no" in resto:
            decision, requiere_revision = "responder", False
        elif "pedí" in texto or "indicá" in texto:
            decision, requiere_revision = "pedir_dato", False
        else:
            decision, requiere_revision = "derivar", True

    return crear_resultado(
        decision=decision,
        respuesta=salida["respuesta"],
        confianza=0.8,
        requiere_revision=requiere_revision,
        traza=[{"paso": "llm", "estado": "ok", "modo": salida["modo"]}],
    )

agente_triage("No puedo ingresar")

---
### 📝 Ejercicio — El agente de tu equipo

Consolidamos todo lo anterior en un único ejercicio: adaptar el agente de triage a las reglas propias de la mesa de ayuda de cada equipo.

Paso 1 — Diseñar la instrucción

- Completá `INSTRUCCION_EQUIPO` (celda siguiente) con rol, objetivo, tres límites, formato de respuesta y condición de derivación:
    - una consulta normal;
    - una que incluya una contraseña (el agente no debe pedirla ni almacenarla).
- Agregar comportamiento:
    - para que decida `pedir_dato` cuando falte un dato necesario (versión, número de ticket, mensaje de error) en vez de responder con suposiciones.


In [ ]:
pregunta = input("Ingresá la consulta para el agente: ")
pregunta

In [ ]:
# Completa el siguiente bloque de instrucciones para que el equipo de desarrollo pueda definir su propio comportamiento del agente de triage:

INSTRUCCION_EQUIPO = """
ROL: ...
OBJETIVO: ...
LÍMITES:
- ...
- ...
- ...
FORMATO:
- RESUMEN:
- SIGUIENTE PASO:
- REQUIERE HUMANO:
DERIVAR CUANDO: ...
""".strip() # funcion que quita los vacios al principio y al final de la cadena de texto

# TODO: reemplazar la respuesta guardada por una salida que respete
# la instrucción diseñada por el equipo.
prueba_equipo = consultar_llm(
    pregunta,
    INSTRUCCION_EQUIPO,
)
prueba_equipo